<div dir="rtl" style="font-family: sans-serif; line-height: 1.6; background-color: #f8f9fa; padding: 20px; border-radius: 8px; border-right: 5px solid #2ecc71;">

# קווסט 3: אתגר הנתונים של חברת התיירות PlaceUp ✈️

מחברת זו מציגה פתרון מקצה לקצה (End-to-End) לאתגר הנתונים של חברת התיירות הבינלאומית **PlaceUp**.
המטרה המרכזית היא לחזות מראש את יעד הטיסה הראשוני של משתמשים חדשים, על מנת לייצר חוויית משתמש מותאמת אישית (Personalization) כבר מרגע הגלישה הראשון, ובכך למנוע נטישת לקוחות באתר.

---

### 📋 חלק א' – תקציר מנהלים ועסקי (Executive Summary)

* **הבעיה העסקית:** הצגת יעדים גנריים ואחידים לכלל המשתמשים מביאה לנטישת האתר (Churn) ואובדן לקוחות פוטנציאליים.
* **הגישה והפתרון:** פיתוח מנוע המלצות מבוסס בינה מלאכותית הלומד את מאפייני הגולש (דמוגרפיה וסוג מכשיר) ומנבא את יעד הטיסה המועדף עליו בזמן אמת.
* **מדדי ההצלחה (KPIs) ושיקולי איזון:** במערכת המאופיינת בחוסר איזון קיצוני בנתונים (רוב הלקוחות אינם מזמינים טיסה כלל - NDF), מדד הדיוק הכללי (Accuracy) הינו מטעה. פרויקט זה מתמקד במדד ה-**F1-Score** ובמדד ה-**Diversity (גיוון המלצות)**, על מנת להבטיח שהמודל לא "יתקע" על המלצה אחידה אלא יספק ערך עסקי וימכור כרטיסי טיסה במציאות.

### 🗺️ מבנה המחברת:
1. **חלק ב' – חקירת נתונים וניקוי (EDA):** טיפול בערכים חסרים וחריגים בעמודת הגיל.
2. **חלק ג' – פיצול שכבתי וקידוד (Pipeline):** מניעת זליגת נתונים (Data Leakage) וביצוע One-Hot Encoding.
3. **חלק ד' – מודל בסיס (Baseline):** בניית מודל רגרסיה לוגיסטית וניתוח כשליו העסקיים.
4. **חלק ה' – מודל מתקדם (Random Forest):** שימוש במשקולות מאוזנות להשגת גיוון עסקי.
5. **חלק ו' – סימולטור אימפקט עסקי (ROI):** תרגום המדדים הסטטיסטיים לשורת הרווח בדולרים.
6. **חלק ז' – הסברתיות (Explainability):** פתיחת הקופסה השחורה וזיהוי הפיצ'רים המובילים.
7. **חלק ח' – סימולציית ייצור (Production API):** קוד פונקציונלי המדמה את פעילות המודל בזמן אמת באתר.

</div>

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# טעינת הקובץ
df = pd.read_csv('PlaceUp_travel_users_dataset.csv')

# הצגת 5 השורות הראשונות של הטבלה
df.head()

<div dir="rtl">

### חלק ב': חקירת נתונים וניקוי (Data Preparation & Exploration)

**היכרות עם הנתונים:**
בשלב הראשון טענו את נתוני המשתמשים של חברת PlaceUp. גילינו שמדובר ב-10,000 רשומות, אך בבחינה של הסטטיסטיקה התיאורית מצאנו שתי בעיות מרכזיות בעמודת הגיל (`age`):
1. **ערכים חסרים:** לכ-800 משתמשים לא הוזן גיל.
2. **ערכים חריגים (Outliers):** מצאנו גילאים לא הגיוניים בעליל, כמו 5- או 2014.

**אסטרטגיית הניקוי:**
מודל בינה מלאכותית רגיש מאוד לנתוני זבל. לכן, הפכנו את כל הגילאים הלא הגיוניים ל"ערכים חסרים", ולאחר מכן השלמנו את כל החוסרים באמצעות **חציון** הגילאים. בחרנו בחציון ולא בממוצע כדי למנוע מצב שבו גילאי הקיצון מעוותים את החישוב.

</div>

In [ ]:
# פקודה ראשונה: נותנת תעודת זהות של הטבלה - סוגי עמודות, כמות שורות וכמות ערכים תקינים (לא חסרים)
print("--- Data Info ---")
df.info()

print("\n--- Summary Statistics ---")
# פקודה שנייה: עושה סטטיסטיקה מהירה (מינימום, מקסימום, ממוצע) על עמודות של מספרים
df.describe()

In [ ]:
import numpy as np # ספרייה שעוזרת לנו לעבוד עם מספרים

print("--- Before Cleaning ---")
print(f"Missing ages: {df['age'].isna().sum()}") # סופר כמה גילאים חסרים

# 1. הופכים גילאים לא הגיוניים ל"חסרים" (NaN)
# אנחנו מגדירים הגיוני כבין 15 ל-120
df.loc[(df['age'] < 15) | (df['age'] > 120), 'age'] = np.nan

# 2. ממלאים את כל הערכים החסרים בחציון (הגיל האמצעי של שאר המשתמשים)
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)

print("\n--- After Cleaning ---")
print(f"Missing ages: {df['age'].isna().sum()}") # מוודאים שלא נשארו חסרים
print(f"Min age: {df['age'].min()}, Max age: {df['age'].max()}") # בודקים את הגילאים החדשים

<div dir="rtl">

### חלק ג' : פיצול חכם והכנת הנתונים (Train-Test Split & Encoding)

**פיצול שכבתי (Stratified Split):**
הנתונים שלנו סובלים מחוסר איזון מובהק (Imbalanced Data) – רוב המשתמשים לא הזמינו טיסה כלל (`NDF`), ורבים אחרים טסו ל-`US`. כדי להבטיח שקבוצת המבחן (Test) שלנו תשקף נאמנה את המציאות ולא תחסיר יעדים נדירים, ביצענו פיצול חכם של 80/20 תוך שימוש ב-Stratified Split השומר על הפרופורציות המקוריות.

**מניעת זליגת נתונים (Data Leakage) וקידוד:**
מודלים מתמטיים אינם יודעים לקרוא טקסט, ולכן המרנו את כל המשתנים הקטגוריאליים (כמו מין או סוג מכשיר) למספרים בינאריים באמצעות `One-Hot Encoding`.
**נקודה קריטית:** פעולה זו בוצעה *אחרי* הפיצול לקבוצות, כדי לוודא שקבוצת האימון לא נחשפת למידע מקבוצת הבדיקה – מה שהיה גורם לזליגת נתונים ומנפח את תוצאות המודל באופן שקרי.

</div>

In [ ]:
from sklearn.model_selection import train_test_split

# 1. נפריד בין המאפיינים (X) לבין יעד הטיסה (y) - נוריד משתנים לא רלוונטיים
X = df.drop(columns=['country_destination', 'user_id'])
y = df['country_destination']

# 2. ביצוע קידוד (One-Hot Encoding) על כלל המאפיינים למניעת אי-תאימות בעמודות
X_encoded = pd.get_dummies(X)

# 3. פיצול שכבתי חכם (Stratified Split) ל-80% אימון ו-20% בדיקה
X_train_encoded, X_test_encoded, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=42,
    stratify=y # שומר על פרופורציית היעדים המקורית
)

print(f"מספר המאפיינים לפני קידוד: {X.shape[1]}")
print(f"מספר המאפיינים לאחר קידוד בסביבת הייצור: {X_train_encoded.shape[1]}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("--- Training Baseline Model (Logistic Regression) ---")

# 1. הגדרת המודל (אנחנו נותנים לו קצת יותר זמן לחשב עם max_iter=1000)
baseline_model = LogisticRegression(max_iter=1000, random_state=42)

# 2. אימון המודל (למידה מה-80% של הנתונים)
# כאן המודל לומד את הקשר בין ה-X (מאפיינים) ל-y (היעד)
baseline_model.fit(X_train_encoded, y_train)

# 3. מבחן המציאות: מבקשים מהמודל לנחש את היעדים של קבוצת הבדיקה (ה-20% שהחבאנו)
y_pred_baseline = baseline_model.predict(X_test_encoded)

# 4. בדיקת ציונים: השוואת הניחושים (y_pred) לתוצאות האמת (y_test)
accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"Baseline Model Accuracy: {accuracy * 100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report

print("--- Detailed Performance Report ---")
# הדו"ח הזה מראה לנו את הביצועים עבור כל מדינה ומדינה!
print(classification_report(y_test, y_pred_baseline, zero_division=0))

<div dir="rtl">

### ניתוח תוצאות: מודל בסיס (Logistic Regression)

מודל ה-Baseline שלנו השיג **Accuracy (דיוק כולל) של 48%**. במבט ראשון זה נראה סביר, אבל דו"ח הסיווג חושף בעיה עסקית חמורה: המודל הפך ל"תלמיד עצלן". במקום ללמוד את ההבדלים בין היעדים, הוא פשוט חוזה את מחלקת הרוב (`NDF` - לא טסים) לכל המשתמשים.

**למה Accuracy הוא מדד מטעה כאן?**
* **Recall (אחוז תפיסה):** המודל תפס 100% מהאנשים שלא טסו (כי הוא ניחש את זה לכולם), אבל ה-Recall שלו ליעדים אחרים כמו ארה"ב (`US`) או צרפת (`FR`) הוא **0.00**.
* **Precision (אמינות):** מתוך כל הפעמים שהמודל טען שמשתמש לא יטוס (`NDF`), הוא צדק רק ב-48% מהפעמים.
* **F1-Score:** זהו המדד המשוקלל והחשוב ביותר כשהנתונים לא מאוזנים. ליעדי הטיסה האמיתיים שלנו הציון הוא **0.00**, **מה שמוכיח שהמודל סובל מחוסר מוחלט של גיוון (Diversity).**

**מסקנה (Smart KPIs):**
הערך העסקי של מודל כזה עבור חברת "PlaceUp" הוא אפס, כיוון שהוא לא מספק שום פרסונליזציה. אנחנו חייבים לעבור למודל מורכב יותר שיידע לאתר תבניות מסובכות יותר ולא ימליץ רק על "ללא יעד" לכולם.

</div>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("--- Training Advanced Model (Random Forest) ---")

# 1. מקימים את "היער":
# n_estimators=100 אומר שאנחנו בונים 100 עצי החלטה.
# class_weight='balanced' - זה קסם! זה אומר למודל: "תקשיב, יש מעט שטסים לאירופה, אז תיתן להם משקל כפול כדי שלא תתעלם מהם".
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)

# 2. אימון המודל (נותנים לו ללמוד מה-80% של קבוצת האימון)
rf_model.fit(X_train_encoded, y_train)

# 3. מבחן המציאות על קבוצת הבדיקה
y_pred_rf = rf_model.predict(X_test_encoded)

# 4. הדפסת התעודה
print("\n--- Random Forest Performance Report ---")
print(classification_report(y_test, y_pred_rf, zero_division=0))

<div dir="rtl">

### השוואת מודלים ושיקולי בחירה (Model Benchmarking)

כעת נשווה בין מודל הבסיס (Logistic Regression) לבין המודל המורכב (Random Forest):

| מדד | Logistic Regression (Baseline) | Random Forest (Advanced) |
| :--- | :--- | :--- |
| **Accuracy (דיוק כללי)** | 48% | 26% |
| **Recall עבור צרפת (FR)** | 0% | 31% |
| **Recall עבור גרמניה (DE)**| 0% | 40% |
| **גיוון המלצות (Diversity)** | נמוך מאוד (חוזה רק NDF) | גבוה (חוזה מגוון יעדים) |

**שיקולי הבחירה למודל הסופי:**
למרות שמודל הבסיס הציג דיוק כללי (Accuracy) גבוה יותר של 48%, הוכחנו כי מדובר במדד מטעה ("אשליית מחלקת הרוב"). המודל לא סיפק שום ערך עסקי כיוון שלא המליץ על טיסות לאירופה או ארה"ב כלל.
לעומתו, מודל ה-**Random Forest** הונחה להתחשב בחוסר האיזון (`class_weight='balanced'`). כתוצאה מכך, הדיוק הכללי ירד, אך המודל החל לבצע קלסיפיקציה אמיתית למגוון יעדים. היכולת לאתר 31% מהנוסעים לצרפת ו-40% מהנוסעים לגרמניה מאפשרת לחברת התיירות להפעיל קמפיינים פרסונליים ולייצר רווח אמיתי. לכן, **ה-Random Forest נבחר כמודל המנצח.**

</div>

<div dir="rtl">
 ⚠️ **הערה חשובה לתצוגה ב-GitHub :** > פרויקט זה כולל רכיבי UI אינטראקטיביים (ipywidgets) כמו סימולטור ROI ו-API בזמן אמת. פלטפורמות גיט מציגות מחברות בצורה סטטית ולכן רכיבים אלו לא יהיו פעילים בתצוגה המקדימה.

 לחוויה מלאה, מומלץ להריץ את המחברת מקומית או לפתוח אותה ב-Google Colab.

</div>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print("--- Loading Interactive Dashboard with Dynamic Insights ---")

# 1. נתוני הבסיס
rf_tp = sum((y_pred_rf != 'NDF') & (y_test == y_pred_rf))
base_tp = sum((y_pred_baseline != 'NDF') & (y_test == y_pred_baseline))
rf_miss = sum(y_test != y_pred_rf)
base_miss = sum(y_test != y_pred_baseline)

# 2. יצירת רכיבי הממשק
style = {'description_width': '150px'}
layout = widgets.Layout(width='400px') # קצת יותר צר כדי שישב יפה ליד הטקסט

profit_slider = widgets.IntSlider(value=200, min=50, max=500, step=25, description='רווח מהזמנה ($):', style=style, layout=layout)
cac_slider = widgets.IntSlider(value=10, min=2, max=50, step=2, description='עלות נטישה/CAC ($):', style=style, layout=layout)
users_slider = widgets.IntSlider(value=10000, min=2000, max=100000, step=2000, description='טראפיק חודשי:', style=style, layout=layout)

# רכיב טקסט דינמי שיתעדכן בקוד
dynamic_insights = widgets.HTML(value="")
out = widgets.Output()

# 3. הפונקציה שמעדכנת הכל יחד
def update_dashboard(profit, cac, users):
    with out:
        clear_output(wait=True)

        multiplier = users / 2000
        base_profit = (base_tp * multiplier * profit) - (base_miss * multiplier * cac)
        rf_profit = (rf_tp * multiplier * profit) - (rf_miss * multiplier * cac)

        profit_diff = rf_profit - base_profit

        # --- עדכון הטקסט הדינמי ---
        insights_html = f"""
        <div dir="rtl" style="background-color: #f4f6f9; padding: 20px; border-radius: 10px; border-right: 6px solid #2ecc71; height: 100%; font-family: Arial, sans-serif; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
            <h3 style="margin-top: 0; color: #2c3e50; font-size: 16px;">💡 התמונה העסקית:</h3>
            <p style="font-size: 14px; color: #555; margin-bottom: 15px;">עם טראפיק של <b>{users:,}</b> משתמשים בחודש:</p>
            <ul style="padding-right: 20px; font-size: 14px; line-height: 1.6;">
                <li>מודל ה-Baseline (ה"בטוח") גורם להפסד של <b style="color: #e74c3c;">${abs(base_profit):,.0f}</b> בגלל נטישת משתמשים.</li>
                <li>מודל ה-Random Forest מתאים יעדים ומייצר <b style="color: #2ecc71;">${rf_profit:,.0f}</b> נטו.</li>
            </ul>
            <div style="margin-top: 15px; padding-top: 15px; border-top: 1px solid #ddd;">
                <span style="font-size: 14px;">האימפקט של ה-AI על החברה:</span><br>
                <b style="font-size: 22px; color: #2ecc71;">+${profit_diff:,.0f} / חודש</b>
            </div>
        </div>
        """
        dynamic_insights.value = insights_html

        # --- ציור הגרף ---
        fig, ax = plt.subplots(figsize=(7, 4.5))
        models = ['Baseline\n(High Churn)', 'Random Forest\n(Personalized)']
        profits = [base_profit, rf_profit]
        colors = ['#e74c3c' if p < 0 else '#2ecc71' for p in profits]

        bars = ax.bar(models, profits, color=colors, width=0.4, edgecolor='none')

        max_val = max(max(profits), 0)
        min_val = min(min(profits), 0)
        padding = (max_val - min_val) * 0.25 if (max_val - min_val) != 0 else 1000
        ax.set_ylim(min_val - padding, max_val + padding)

        for bar in bars:
            yval = bar.get_height()
            offset = padding * 0.1
            if yval >= 0:
                ax.text(bar.get_x() + bar.get_width()/2, yval + offset, f'${yval:,.0f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
            else:
                ax.text(bar.get_x() + bar.get_width()/2, yval - offset, f'${yval:,.0f}', ha='center', va='top', fontsize=12, fontweight='bold')

        ax.axhline(0, color='#2c3e50', linewidth=1.2)
        ax.set_title('Net Profit Simulator', fontsize=14, fontweight='bold', pad=15)

        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ',')))
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='y', linestyle='--', alpha=0.4)

        plt.tight_layout()
        plt.show()

# 4. סידור הוויזואליה: סליידרים בצד אחד, תיבת תובנות בצד שני!
controls = widgets.VBox([profit_slider, cac_slider, users_slider], layout=widgets.Layout(justify_content='center', padding='10px'))
top_panel = widgets.HBox([controls, dynamic_insights], layout=widgets.Layout(align_items='stretch', margin='0 0 20px 0'))

widgets.interactive_output(update_dashboard, {'profit': profit_slider, 'cac': cac_slider, 'users': users_slider})
display(widgets.VBox([top_panel, out]))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Unlocking the Black Box: Feature Importance ---")

# 1. שליפת חשיבות הפיצ'רים מהמודל המאומן שלנו
importances = rf_model.feature_importances_
features = X_train_encoded.columns

# 2. סידור הנתונים בטבלה ולקיחת ה-10 הכי חשובים
feat_imp_df = pd.DataFrame({'Feature': features, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False).head(10)

# 3. ציור הגרף (עיצוב תאגידי נקי)
plt.figure(figsize=(10, 6))
# משתמשים בצבע מודרני ואחיד כדי לא לייצר "קרקס" בעין
bars = sns.barplot(x='Importance', y='Feature', data=feat_imp_df, color='#3498db', edgecolor='none')

# 4. הוספת ערכים מספריים על כל עמודה
for p in bars.patches:
    width = p.get_width()
    plt.text(width + 0.002, p.get_y() + p.get_height()/2, f'{width:.3f}',
             ha='left', va='center', fontsize=11, color='#2c3e50', fontweight='bold')

# 5. עיצוב סופי
plt.title('Top 10 Most Influential Features (What drives user decisions?)', fontsize=16, fontweight='bold', pad=20, color='#2c3e50')
plt.xlabel('Importance Score (Relative impact on the final decision)', fontsize=12, color='#7f8c8d')
plt.ylabel('User Characteristic', fontsize=12, color='#7f8c8d')

# הסרת מסגרות מיותרות
sns.despine(left=True, bottom=True)
plt.tick_params(axis='both', length=0) # הסרת קווי השנתות הקטנים

plt.tight_layout()
plt.show()

<div dir="rtl" style="font-family: sans-serif; line-height: 1.6;">

### סעיף 3: הסברתיות (Explainability) - פתיחת "הקופסה השחורה"

כדי לוודא שהמודל שלנו לא רק מספק תחזיות אלא גם מובן עסקית להנהלה, חילצנו את חשיבות הפיצ'רים (Feature Importance) מתוך מודל ה-Random Forest.

**תובנות עסקיות מרכזיות מתוך הגרף:**
1. **הגיל (Age) הוא המנבא החזק ביותר:** עם משקל השפעה של למעלה מ-60%, הגיל הוא הגורם המכריע ביותר. זה מצביע על כך שלקבוצות גיל שונות (סטודנטים, משפחות, פנסיונרים) יש דפוסי הזמנה מובהקים ושונים לחלוטין.
2. **תהליך ההרשמה (Signup Flow):** הפיצ'ר השני בחשיבותו מרמז שהמסלול או דף הנחיתה שדרכו הגיע המשתמש, מעיד רבות על כוונת הרכישה (Intent) שלו.
3. **סוג המכשיר (Mac Desktop):** נתון מעניין הוא שמשתמשי מק משפיעים על המודל יותר ממשתמשי Windows או אייפון. זה עשוי להעיד על פלח שוק בעל הכנסה פנויה גבוהה יותר שנוטה לסגור סוג מסוים של חופשות.

**המלצה אופרטיבית לחברת PlaceUp:**
צוות השיווק חייב לבנות את אסטרטגיית הפרסונליזציה שלו קודם כל סביב פילוחי גיל (Age Segments), ולהתאים את המסרים והתמחור בהתאם לפלטפורמה ממנה גולש המשתמש.

</div>

<div dir="rtl" style="font-family: sans-serif; line-height: 1.6;">

### בונוס: הדמיית המודל בסביבת ייצור (Production Simulation) 🚀

מודל Machine Learning לא שווה הרבה אם הוא נשאר רק בתוך מחברת המחקר. כדי להדגים כיצד פרויקט זה משתלב במוצר האמיתי של PlaceUp, בניתי סימולטור End-to-End המדמה כניסה של משתמשים בזמן אמת לאתר:

**מה אנחנו רואים בסימולציה למטה?**
המסך מפוצל לשני חלקים המדמים את מחזור החיים המלא של ההמלצה:
1. **צד השרת (Backend - שחור):** המנוע מקבל משתמש גולמי, מנקה את הנתונים בזמן ריצה (למשל, טיפול בגילאים חסרים), ומפעיל את מודל ה-Random Forest. המערכת חכמה מספיק להפעיל מנגנון **Fallback**: אם המודל מנבא `NDF` (חוסר יעד), אנחנו שולפים את ההסתברות השנייה בגובהה כדי לא לאבד את הלקוח.
2. **צד הלקוח (Frontend - לבן):** המערכת לוקחת את תחזית היעד ואת גיל המשתמש (הפיצ'ר החזק ביותר שזיהינו), ומייצרת בזמן אמת **קופי שיווקי מותאם אישית (Push Notification)** לדף הבית.

*כך הופכים דאטא ואלגוריתמיקה לחוויית משתמש (UX) שמייצרת לחברה כסף.*

</div>

In [ ]:
import pandas as pd
import json
from IPython.display import display, HTML

print("--- Initializing End-to-End Production Simulator ---")

# מילון תרגום יעדים כדי שחווית המשתמש תהיה אמיתית
DEST_MAP = {
    'US': 'ארה"ב 🇺🇸', 'FR': 'צרפת 🇫🇷', 'ES': 'ספרד 🇪🇸',
    'IT': 'איטליה 🇮🇹', 'GB': 'בריטניה 🇬🇧', 'DE': 'גרמניה 🇩🇪',
    'NL': 'הולנד 🇳🇱', 'AU': 'אוסטרליה 🇦🇺', 'PT': 'פורטוגל 🇵🇹',
    'other': 'יעד אקזוטי 🌍', 'NDF': 'ללא יעד'
}

class PlaceUpProductionEngine:
    def __init__(self, trained_model, encoded_columns_template):
        self.model = trained_model
        self.columns_template = encoded_columns_template

    def _preprocess(self, raw_user):
        df = pd.DataFrame([raw_user])
        if 'age' in df.columns:
            age_val = df['age'].values[0]
            if pd.isna(age_val) or age_val < 14 or age_val > 100:
                df['age'] = 34

        df_encoded = pd.get_dummies(df)
        df_final = pd.DataFrame(0, index=[0], columns=self.columns_template)
        for col in df_encoded.columns:
            if col in df_final.columns:
                df_final[col] = df_encoded[col]

        return df_final, df['age'].values[0]

    def process_user(self, raw_user):
        processed_data, age = self._preprocess(raw_user)
        probs = self.model.predict_proba(processed_data)[0]
        classes = self.model.classes_

        dest_probs = sorted(zip(classes, probs), key=lambda x: x[1], reverse=True)
        primary_pred, confidence = dest_probs[0][0], dest_probs[0][1]

        # לוגיקת Fallback ותרגום חזותי
        best_dest = dest_probs[0][0] if dest_probs[0][0] != 'NDF' else dest_probs[1][0]
        pretty_dest = DEST_MAP.get(best_dest, best_dest)

        # יצירת קופי שיווקי
        if primary_pred == 'NDF':
            headline = "מחפשים השראה לחופשה? ✨"
            msg = f"עדיין מתלבטים? מצאנו דילים חמים של הרגע האחרון ל{pretty_dest} שמתאימים בדיוק לפרופיל שלכם!" if age < 35 else f"מגיע לכם להתפנק. חבילות נופש שקטות ל{pretty_dest} מחכות לכם עכשיו באתר."
        else:
            headline = f"היעד הבא שלך: {pretty_dest}!"
            msg = f"ארזתם כבר? חבילות צעירים ל{pretty_dest} עם חיי לילה מטורפים והנחות מיוחדות." if age < 35 else f"גלו את הקסם של {pretty_dest} עם סיורים תרבותיים ומלונות בוטיק יוקרתיים."

        # ה-Payload המלא
        payload = {
            "status": "success",
            "user_id": raw_user.get("user_id"),
            "model_insights": {
                "math_prediction": primary_pred,
                "confidence_score": f"{confidence * 100:.1f}%",
                "fallback_destination": best_dest
            },
            "ui_payload": {
                "headline": headline,
                "notification_text": msg
            }
        }

        # בניית חווית ה-UI ב-HTML
        self._render_simulation(raw_user, payload)

    def _render_simulation(self, user_data, payload):
        """פונקציה שמייצרת ממשק ויזואלי מפוצל - שרת מול לקוח"""
        json_str = json.dumps(payload, indent=4, ensure_ascii=False)

        html = f"""
        <div style="font-family: Arial, sans-serif; margin-bottom: 30px; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">
            <div style="background: #2c3e50; color: white; padding: 10px 20px; font-size: 16px; font-weight: bold; direction: ltr;">
                👤 User Event Triggered: {user_data['user_id']} | Age: {user_data['age']} | Device: {user_data['first_device_type']}
            </div>

            <div style="display: flex; flex-wrap: wrap; background: #fff;">

                <div style="flex: 1; min-width: 300px; background: #1e1e1e; padding: 20px; color: #d4d4d4;">
                    <h3 style="color: #4ec9b0; margin-top: 0; font-size: 15px;">⚙️ Backend API Response (JSON)</h3>
                    <pre dir="ltr" style="font-family: Consolas, monospace; font-size: 12px; overflow-x: auto;">{json_str}</pre>
                </div>

                <div style="flex: 1; min-width: 300px; padding: 20px; background: #f8f9fa; direction: rtl;">
                    <h3 style="color: #3498db; margin-top: 0; font-size: 15px;">📱 חווית המשתמש באתר (Frontend UI)</h3>

                    <div style="background: white; border: 1px solid #e1e8ed; border-radius: 10px; padding: 15px; margin-top: 15px; position: relative;">
                        <div style="display: flex; align-items: center; margin-bottom: 10px;">
                            <div style="background: #e74c3c; width: 40px; height: 40px; border-radius: 50%; display: flex; align-items: center; justify-content: center; color: white; font-weight: bold; font-size: 20px; margin-left: 15px;">P</div>
                            <div>
                                <h4 style="margin: 0; color: #14171a; font-size: 16px;">PlaceUp - התאמה אישית</h4>
                                <span style="font-size: 12px; color: #657786;">עכשיו</span>
                            </div>
                        </div>
                        <h3 style="margin: 0 0 5px 0; color: #1da1f2; font-size: 18px;">{payload['ui_payload']['headline']}</h3>
                        <p style="margin: 0; color: #14171a; font-size: 14px; line-height: 1.5;">{payload['ui_payload']['notification_text']}</p>

                        <div style="margin-top: 15px;">
                            <button style="background: #1da1f2; color: white; border: none; padding: 8px 15px; border-radius: 20px; font-weight: bold; cursor: pointer;">צפה בדילים</button>
                        </div>
                    </div>
                </div>

            </div>
        </div>
        """
        display(HTML(html))

# --- הרצת הסימולציה ---
engine = PlaceUpProductionEngine(trained_model=rf_model, encoded_columns_template=X_train_encoded.columns)

user_young = {"user_id": "USR_9941", "age": 22, "gender": "MALE", "first_device_type": "iPhone"}
user_senior = {"user_id": "USR_1102", "age": 55, "gender": "FEMALE", "first_device_type": "Mac Desktop"}

engine.process_user(user_young)
engine.process_user(user_senior)

<div dir="rtl" style="font-family: sans-serif; line-height: 1.6;">

### סעיף 5: שאלות פתוחות ותכנון קדימה

**1. תובנות מהמחקר הראשוני של הדאטה:**
הדבר הראשון שקפץ לעין כשחקרתי את הנתונים בהתחלה זה חוסר האיזון (Imbalance) הקיצוני. רוב המשתמשים לא סגרו טיסה (NDF). זה ישר הדליק לי נורה אדומה שמודל פשוט "ירמה" וינחש כל הזמן NDF כדי לקבל Accuracy גבוה (וזה בדיוק מה שקרה במודל ה-Baseline). בגלל זה היה קריטי לעבור למודל Random Forest ולהכריח אותו להתחשב במשקלים.
עוד משהו שבלט זה הגיל. כבר בניקוי הנתונים ראינו שונות גדולה, ובסוף גם הוכחנו את זה דרך ה-Feature Importance - הגיל הוא חד משמעית הנתון שהכי משפיע על המודל שלנו (מעל 60% חשיבות).

**2. איזה דאטה נוסף היה יכול לשפר את המודל?**
כדי לקחת את המערכת הזו לשלב הבא ולהקפיץ את אחוזי ההמרה (Conversion), חסר פה המון דאטה התנהגותי (Clickstream).
כרגע אנחנו מסתמכים בעיקר על דמוגרפיה וסוג מכשיר, אבל בעולם האמיתי הייתי רוצה לראות:
* **התנהגות באתר:** כמה זמן משתמש שהה באתר? על איזה מדינות הוא בכלל הסתכל לפני שנטש?
* **העדפות**: איזה סוגי טיסות ומדינות הלקוח מעדיף? עירות נופש, אקסטרים, פעילות וכו.
* **היסטוריה:** האם זה לקוח חוזר? לאן הוא טס בשנה שעברה?
* **תקציב (Budget):** אם היה לנו איזשהו מדד של רגישות למחיר, יכולנו לא רק להמליץ על ארה"ב או איטליה, אלא גם להתאים את רמת המלון והטיסה ליכולת הכלכלית שלו.

</div>